**Purpose of the Project**

*The purpose of this project is to build a Recurrent Neural Network (RNN) that can understand and classify text data automatically. The model learns from a large collection of text samples and predicts the correct category or label for new text. This project demonstrates how deep learning can be used for Natural Language Processing (NLP) tasks such as text classification, making it easier to analyze large amounts of textual data quickly and accurately.*


In [34]:
pip install datasets

In [35]:
from datasets import load_dataset

# Use the full namespace/dataset format
dataset = load_dataset("wangrongsheng/ag_news", split="train")

In [36]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split


**Step 2: Clean and Normalize the Text**

In [37]:
import re

def clean_my_text(text):
    text = text.lower()                   # Convert to Lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Delete symbols and numbers
    text = re.sub(r'\s+', ' ', text)      # Remove the extra spaces
    return text.strip()

# Cleaned Dataset
clean_dataset = dataset.map(lambda x: {"text": clean_my_text(x["text"])})

**Step 3: Tokenize the Strings into Numbers**

In [38]:
texts = []

for i, item in enumerate(clean_dataset):
    texts.append(item["text"])
    if i == 19999:  # Use the first 20,000 samples
        break

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

word_index = tokenizer.word_index

print(word_index)
print("Vocabulary Size:", len(word_index))

{'<OOV>': 1, 'the': 2, 'to': 3, 'a': 4, 'of': 5, 'in': 6, 'and': 7, 'on': 8, 'for': 9, 's': 10, 'that': 11, 'with': 12, 'at': 13, 'as': 14, 'us': 15, 'new': 16, 'its': 17, 'by': 18, 'is': 19, 'reuters': 20, 'it': 21, 'said': 22, 'from': 23, 'an': 24, 'ap': 25, 'has': 26, 'his': 27, 'after': 28, 'was': 29, 'will': 30, 'have': 31, 'be': 32, 'over': 33, 'their': 34, 'are': 35, 'up': 36, 'but': 37, 'two': 38, 'more': 39, 'first': 40, 'oil': 41, 'he': 42, 'this': 43, 'athens': 44, 'monday': 45, 'out': 46, 'not': 47, 'olympic': 48, 'one': 49, 'york': 50, 'thursday': 51, 'friday': 52, 'tuesday': 53, 'world': 54, 'company': 55, 'into': 56, 'iraq': 57, 'wednesday': 58, 'gold': 59, 'about': 60, 'than': 61, 'they': 62, 'were': 63, 'last': 64, 'inc': 65, 'who': 66, 'prices': 67, 'against': 68, 'yesterday': 69, 'been': 70, 'had': 71, 'week': 72, 'million': 73, 'year': 74, 'when': 75, 'sunday': 76, 'says': 77, 'president': 78, 'united': 79, 'government': 80, 'people': 81, 'microsoft': 82, 'afp': 83,

**Step 4: Pad and Truncate Sequences**

In [39]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Convert text to integer sequences
sequences = tokenizer.texts_to_sequences(texts)

# Pad and truncate sequences to a fixed length of 50
max_length = 50

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print(padded_sequences.shape)
print(padded_sequences[0])

(20000, 50)
[  422   784  2211  8422   113    56     2   875    20    20 15969   422
  1702 21253     5 15970    35  3629  1305   332     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


**Step 5: Convert Labels to Categorical Vectors**

In [40]:
labels = clean_dataset["label"]

In [41]:
# Convert labels to a NumPy array
labels = np.array(labels)

# One-hot encode the labels
categorical_labels = to_categorical(labels)

print(categorical_labels.shape)
print(categorical_labels[:5])

(120000, 4)
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


**Step 6: Define the RNN Model Architecture**

In [42]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128
max_length = 50
num_classes = categorical_labels.shape[1]

model = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              input_length=max_length),

    SimpleRNN(64),

    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 7: Compile and Train the Model**

In [43]:
model.compile(
    optimizer=Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    padded_sequences,
    categorical_labels,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 47s 79ms/step - accuracy: 0.6143 - loss: 0.8802 - val_accuracy: 0.7172 - val_loss: 0.7876
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.8436 - loss: 0.4704 - val_accuracy: 0.7305 - val_loss: 0.7349
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.9079 - loss: 0.3016 - val_accuracy: 0.7402 - val_loss: 0.8050
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 66ms/step - accuracy: 0.9334 - loss: 0.2202 - val_accuracy: 0.6963 - val_loss: 0.9909
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - accuracy: 0.9480 - loss: 0.1759 - val_accuracy: 0.6985 - val_loss: 1.0819
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 71ms/step - accuracy: 0.9523 - loss: 0.1596 - val_accuracy: 0.7290 - val_loss: 1.0708
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 43s 76ms/step - accuracy: 0.9331 - loss: 0.1905 - val_accuracy: 0.7090 - val_loss: 1.1807
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 77ms/step - accuracy: 0.9718 - loss: 0.1020 - 

**Step 8: Read and Evaluate the Target Accuracy**

In [44]:
labels = clean_dataset["label"][:20000]

labels = np.array(labels)
categorical_labels = to_categorical(labels)

In [45]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    categorical_labels,
    test_size=0.2,
    random_state=42
)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9310 - loss: 0.2888
Test Loss: 0.28884071111679077
Test Accuracy: 0.9309999942779541
